In [1]:
from pathlib import Path
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer


c:\Users\shravan\Documents\OMNI_RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
BASE_DIR = Path("..")
DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

FAISS_INDEX_PATH = PROCESSED_DIR / "faiss.index"
FAISS_META_PATH = PROCESSED_DIR / "faiss_chunks_meta.pkl"


In [3]:
faiss_index = faiss.read_index(str(FAISS_INDEX_PATH))
print("FAISS index loaded")
print("Total vectors:", faiss_index.ntotal)


RuntimeError: Error in __cdecl faiss::FileIOReader::FileIOReader(const char *) at D:\a\faiss-wheels\faiss-wheels\third-party\faiss\faiss\impl\io.cpp:70: Error: 'f' failed: could not open ..\data\processed\faiss.index for reading: No such file or directory

In [ ]:
assert faiss_index.ntotal > 0


In [ ]:
with open(FAISS_META_PATH, "rb") as f:
    faiss_nodes = pickle.load(f)

print("Metadata entries:", len(faiss_nodes))


Metadata entries: 23


In [ ]:
assert len(faiss_nodes) == faiss_index.ntotal


In [ ]:
embed_model = SentenceTransformer("BAAI/bge-base-en")
EMBED_DIM = embed_model.get_sentence_embedding_dimension()

print("Embedding dim:", EMBED_DIM)


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 473.66it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: BAAI/bge-base-en
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding dim: 768


In [ ]:
assert EMBED_DIM == faiss_index.d


In [ ]:
def faiss_search(query: str, k: int = 5):
    query_vec = embed_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    scores, indices = faiss_index.search(query_vec, k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        node = faiss_nodes[idx]
        results.append({
            "score": float(score),
            "text": node.text[:500],
            "metadata": node.metadata
        })

    return results


In [ ]:
query = "environment setup for OmniRAG"

results = faiss_search(query, k=5)

for r in results:
    print("=" * 80)
    print("Score:", r["score"])
    print(r["text"])
    print(r["metadata"])


Score: 0.7412197589874268
13. How do you quantify performance improvements? Metrics include inference latency, throughput, memory footprint, and model response coherence
compared to baseline metrics. 14. What are typical results achieved? Latency reduced by ~47%, context window extended 1.65×, and GPU utilization improved by 26%
after tuning. 15. How do you log and reproduce experiments? All configurations, metrics, and timestamps are stored in JSON files under data/tuned_results.json
for reproducibility. 16. How could t
{'source_type': 'pdf', 'file_name': 'Bliss_LLM_Interview_QA.pdf', 'file_path': 'c:\\Users\\shravan\\Documents\\OMNI_RAG\\notebooks\\..\\data\\raw\\Bliss_LLM_Interview_QA.pdf', 'ingested_at': '2026-02-09T19:09:07.276657', 'content_hash': '04c61b10dbd899fbc373a379ab28e861', 'chunk_id': 1, 'embedding_model': 'bge-base-en', 'chunk_strategy': 'semantic_sentence_merge'}
Score: 0.7364625334739685
What is Hugging Face Transformers library and how have you used it? Answer: It's

In [ ]:
# import chromadb

# chroma_client = chromadb.Client()
# collection = chroma_client.get_collection("omnrag_chunks")

# print("Chroma collection loaded")


In [ ]:
# def chroma_search(query: str, k: int = 5, where: dict | None = None):
#     query_vec = embed_model.encode(
#         query,
#         normalize_embeddings=True
#     ).tolist()

#     results = collection.query(
#         query_embeddings=[query_vec],
#         n_results=k,
#         where=where
#     )

#     output = []
#     for i in range(len(results["documents"][0])):
#         output.append({
#             "text": results["documents"][0][i][:500],
#             "metadata": results["metadatas"][0][i],
#             "score": results["distances"][0][i]
#         })

#     return output


In [ ]:
# query = "OmniRAG environment setup"

# print("\n--- FAISS RESULTS ---\n")
# for r in faiss_search(query):
#     print(r["score"])
#     print(r["text"][:200])
#     print()

# print("\n--- CHROMA RESULTS ---\n")
# for r in chroma_search(query):
#     print(r["score"])
#     print(r["text"][:200])
#     print()


In [ ]:
SIM_THRESHOLD = 0.7

def gated_faiss_search(query: str, k: int = 2):
    results = faiss_search(query, k)
    filtered = [r for r in results if r["score"] >= SIM_THRESHOLD]
    return filtered


In [ ]:
gated_results = gated_faiss_search(query)

if not gated_results:
    print("❌ No confident results — do NOT answer")
else:
    print("✅ Confident retrieval")
    for r in gated_results:
        print(r["score"], r["text"][:200])


✅ Confident retrieval
0.7412197589874268 13. How do you quantify performance improvements? Metrics include inference latency, throughput, memory footprint, and model response coherence
compared to baseline metrics. 14. What are typical resul
0.7364625334739685 What is Hugging Face Transformers library and how have you used it? Answer: It's a Python library to work with pretrained NLP models. I've used it to build
question-answering, text generation, and RAG
0.7334008812904358 16.
0.7215436697006226 1. What is the core idea behind Bliss-LLM? Bliss-LLM is a hardware-aware auto-tuning engine that optimizes LLM inference parameters like
temperature, top_p, and precision using Bayesian optimization t
0.7192965149879456 Python Interview Q&A for Data Science and Gen AI
Generated on: June 25, 2025
Experience Level: 2+ Years
1. What are Python's key features that make it suitable for data science and Gen AI development?


In [ ]:
def debug_query(query: str):
    results = faiss_search(query, k=10)
    for r in results:
        print(r["score"], r["metadata"].get("file_name"))
